# Pacemaker

In [1]:
import pandas as pd
from pathlib import Path

all_es = pd.read_parquet('all_es_combined.parquet')

In [2]:
# Create a new DataFrame called 'a4c_df' by filtering 'all_es'
# all_es = combined_df
a4c_df = all_es[all_es['pred_view'] == 'a4c'].copy()

# This regex finds 'echo-study', 'echo-study-1', or 'echo-study-2',
# and then captures the sequence of characters that follows until the next slash.
regex_pattern = r'results/echo-study(?:-[12])?/([^/]+)'

# Use .str.extract() to pull out the captured group (the Study ID)
a4c_df['DeidentifiedStudyID'] = a4c_df['png_uri'].str.extract(regex_pattern)

a4c_df = a4c_df[['DeidentifiedStudyID', 'mp4_uri_corrected']].rename(
    columns={'mp4_uri_corrected': 'URI'}
)

In [3]:
print(a4c_df.shape)
display(a4c_df.head())

(2976025, 2)


,DeidentifiedStudyID,URI
3,1.2.276.0.7230010.3.1.2.1714512485.1.170311135...,s3://echodata25/results/echo-study/1.2.276.0.7...
5,1.2.276.0.7230010.3.1.2.1714512485.1.170311985...,s3://echodata25/results/echo-study/1.2.276.0.7...
9,1.2.276.0.7230010.3.1.2.1714512485.1.170311135...,s3://echodata25/results/echo-study/1.2.276.0.7...
11,1.2.276.0.7230010.3.1.2.1714512485.1.170311135...,s3://echodata25/results/echo-study/1.2.276.0.7...
13,1.2.276.0.7230010.3.1.2.1714512485.1.170311306...,s3://echodata25/results/echo-study/1.2.276.0.7...


# Connect to Labels

In [4]:
pacemaker_labels = pd.read_csv('pacemaker_labels.csv')

In [6]:
# Select only the necessary columns from your labels DataFrame for efficiency
labels_to_merge = pacemaker_labels[['DeidentifiedStudyID', 'Value']]

# Merge the two DataFrames to find the overlap and add the 'Value'
# 'how="inner"' ensures only matching DeidentifiedStudyIDs are kept
a4c_pacemaker_labels = pd.merge(
    a4c_df, 
    labels_to_merge, 
    on='DeidentifiedStudyID', 
    how='inner'
)

print(f"Found {len(a4c_pacemaker_labels)} overlapping studies.")
a4c_pacemaker_labels.head()

Found 270651 overlapping studies.


,DeidentifiedStudyID,URI,Value
0,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present
1,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present
2,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present
3,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present
4,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present


In [7]:
a4c_pacemaker_labels.to_csv('a4c_pacemaker_labels.csv')

In [8]:
a4c_pacemaker_labels.head()

,DeidentifiedStudyID,URI,Value
0,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present
1,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present
2,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present
3,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present
4,1.2.276.0.7230010.3.1.2.1714512485.1.170311761...,s3://echodata25/results/echo-study/1.2.276.0.7...,present


In [9]:
import pandas as pd  
from sklearn.model_selection import train_test_split  
  
# Read mitral valve regurgitation labels for A4C videos
df = pd.read_csv('a4c_pacemaker_labels.csv')  
  
# Create label mapping for severity levels  
label_mapping = {  
    'absent': 0,  
    'present': 1
}  

def build_train_test(df, label_mapping):  
    # Convert string labels to integers  
    df['Value_numeric'] = df['Value'].map(label_mapping)  
      
    # Verify all labels were mapped correctly  
    if df['Value_numeric'].isna().any():  
        print("Warning: Some labels couldn't be mapped!")  
        print("Unmapped labels:", df[df['Value_numeric'].isna()]['Value'].unique())  
      
    # First split: separate out test set (60% train+val, 40% test)  
    train_val_df, test_df = train_test_split(  
        df,   
        test_size=0.2,  # 20% for test  
        stratify=df['Value_numeric'],   
        random_state=42  
    )  
      
    # Second split: split train+val into train and validation (64% train, 16% val, 20% test)  
    train_df, val_df = train_test_split(  
        train_val_df,  
        test_size=0.2,  # 20% of remaining 80% = 16% of total  
        stratify=train_val_df['Value_numeric'],  
        random_state=42  
    )  

    return train_df, val_df, test_df

In [11]:
train_df, val_df, test_df = build_train_test(df, label_mapping)

# Save files with URI and numeric labels (as expected by V-JEPA 2)  
train_df[['URI', 'Value_numeric']].to_csv('a4c_pacemaker_labels_train.csv', header=False, index=False, sep=' ')  
val_df[['URI', 'Value_numeric']].to_csv('a4c_pacemaker_labels_val.csv', header=False, index=False, sep=' ')  
test_df[['URI', 'Value_numeric']].to_csv('a4c_pacemaker_labels_test.csv', header=False, index=False, sep=' ')  
  
# Print split statistics  
print(f"Total samples: {len(df)}")  
print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")  
print(f"Validation: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")  
print(f"Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")  
  
# Print label distribution for each split  
print("\nLabel distribution:")  
for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:  
    print(f"{split_name}:")  
    for label, count in split_df['Value_numeric'].value_counts().sort_index().items():  
        original_label = [k for k, v in label_mapping.items() if v == label][0]  
        print(f"  {label} ({original_label}): {count}")  

Total samples: 270651
Train: 173216 (64.0%)
Validation: 43304 (16.0%)
Test: 54131 (20.0%)

Label distribution:
Train:
  0 (absent): 79677
  1 (present): 93539
Val:
  0 (absent): 19919
  1 (present): 23385
Test:
  0 (absent): 24900
  1 (present): 29231


# Make a Tiny Subset

In [12]:
import numpy as np
import pandas as pd

def sample_per_class(df: pd.DataFrame,
                     label_col: str = "Value_numeric",
                     n_per_class: int = 500,
                     seed: int = 0) -> pd.DataFrame:
    """Balanced down-sample for fast overfit/debug runs."""
    rng = np.random.default_rng(seed)
    return (
        df.groupby(label_col, group_keys=False)
          .apply(lambda g: g.sample(min(len(g), n_per_class), random_state=rng.integers(1e9)))
          .reset_index(drop=True)
    )

# --- choose sizes ----------------------------------------------------
train_small = sample_per_class(train_df, n_per_class=1000)   # 2 000 rows (1 000 × 2 classes)
val_small   = sample_per_class(val_df,   n_per_class=250)    #   500 rows
test_small  = sample_per_class(test_df,  n_per_class=250)    #   500 rows

# --- save in V-JEPA 2 format ----------------------------------------
for name, df_small in [("train", train_small),
                       ("val",   val_small),
                       ("test",  test_small)]:
    df_small[["URI", "Value_numeric"]].to_csv(
        f"a4c_pacemaker_labels_{name}_tiny.csv",
        header=False, index=False, sep=" "
    )

# quick sanity check
for split, d in [("train", train_small), ("val", val_small), ("test", test_small)]:
    counts = d["Value_numeric"].value_counts().sort_index().to_dict()
    print(split, len(d), counts)

train 2000 {0: 1000, 1: 1000}
val 500 {0: 250, 1: 250}
test 500 {0: 250, 1: 250}


/tmp/ipykernel_46151/442783274.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), n_per_class), random_state=rng.integers(1e9)))
/tmp/ipykernel_46151/442783274.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), n_per_class), random_state=rng.integers(1e9)))
/tmp/ipykernel_46151/442783274.py:12: DeprecationWarning: DataFrameGroup